# Inspect a saved query niche
Enter a **center cell name**, **slice name**, and **niche dimension**, then run the cells in order. The example selects a large niche. Target cell counts come from `experiment/manifests/query_niche_dimensions.json`. A target is a minimum: the saved neighborhood can contain more cells. Only a row with the requested `target_niche_size` and `target_size_reached=True` is accepted.

The figure highlights the exact `niche_cell_names` saved in the metrics CSV, with a star for the center. Outputs include the original metrics as CSV columns, center coordinates, a member-cell table, a parcellation composition table, and a PNG.

**Colab:** make the repository's `experiment/manifests/query_niche_dimensions.json`, `experiment/query_niche_metrics` and matching `data/20260601_225717` H5AD files accessible in Google Drive (or upload them), and set `PROJECT_ROOT` below. Colab cannot directly access a path on your local computer. CSVs are streamed; expression matrices are not loaded. Cell IDs are always treated as strings.


In [ ]:
# Install the notebook's dependencies in Colab; skip if already available locally.
import importlib.util
import subprocess
import sys
packages = ['anndata', 'numpy', 'pandas', 'matplotlib', 'IPython']
missing = [name for name in packages if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


In [ ]:
# Optional: mount Drive when using Colab.
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# Inputs. For Colab, use e.g. /content/drive/MyDrive/NicheQueryBenchmark.
PROJECT_ROOT = "/home/sheryl/niche_query/NicheQueryBenchmark" #@param {type:"string"}
CENTER_CELL_NAME = "1019171910101420741" #@param {type:"string"}
SLICE_NAME = "C57BL6J-638850.28" #@param {type:"string"}
NICHE_DIMENSION = "large" #@param ["large", "median", "small"]
# Relative paths are resolved against PROJECT_ROOT.
METRICS_DIR = f"experiment/query_niche_metrics/{NICHE_DIMENSION}"
DATA_DIR = "data/20260601_225717" #@param {type:"string"}
OUTPUT_DIR = "experiment/niche_visualizations" #@param {type:"string"}
SPATIAL_KEY = "spatial" #@param {type:"string"}
INVERT_Y_AXIS = False #@param {type:"boolean"}


In [ ]:
import csv
import json
import re
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Some niche membership fields exceed Python's default CSV field-size limit.
csv.field_size_limit(100_000_000)
root = Path(PROJECT_ROOT).expanduser()
with (root / "experiment/manifests/query_niche_dimensions.json").open(encoding="utf-8") as handle:
    QUERY_NICHE_DIMENSIONS = json.load(handle)
TARGET_SIZE = QUERY_NICHE_DIMENSIONS[NICHE_DIMENSION]
def resolve_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else root / path

def find_niche(csv_path, center_name, target_size):
    if not csv_path.is_file():
        raise FileNotFoundError(f'Metrics CSV not found: {csv_path}')
    seen = []
    with csv_path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        required = {'center_cell_name', 'target_niche_size', 'target_size_reached',
                    'niche_cell_names', 'niche_cell_count'}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Missing CSV columns: {sorted(missing)}')
        for row in reader:
            if row['center_cell_name'] != center_name:
                continue
            seen.append((row['target_niche_size'], row['target_size_reached']))
            if (int(row['target_niche_size']) == target_size
                    and row['target_size_reached'].strip().lower() == 'true'):
                return row
    if seen:
        raise ValueError(f'Center found, but no reached niche for target {target_size}. '
                         f'Available (target, reached): {seen}')
    raise ValueError(f'Center {center_name!r} was not found in {csv_path.name}.')

CENTER_CELL_NAME = str(CENTER_CELL_NAME).strip()
SLICE_NAME = SLICE_NAME.strip()
if not CENTER_CELL_NAME or not SLICE_NAME or Path(SLICE_NAME).name != SLICE_NAME:
    raise ValueError('Enter a center cell name and a slice filename stem without directories.')
if isinstance(TARGET_SIZE, bool) or not isinstance(TARGET_SIZE, int) or TARGET_SIZE < 1:
    raise ValueError('TARGET_SIZE must be a positive integer.')
metrics_path = resolve_path(METRICS_DIR) / f'{SLICE_NAME}.csv'
h5ad_path = resolve_path(DATA_DIR) / f'{SLICE_NAME}.h5ad'
row = find_niche(metrics_path, CENTER_CELL_NAME, TARGET_SIZE)
member_names = json.loads(row['niche_cell_names'])
if not isinstance(member_names, list) or not all(isinstance(x, str) for x in member_names):
    raise ValueError('niche_cell_names must be a JSON array of string cell IDs.')
if len(set(member_names)) != len(member_names):
    raise ValueError('The saved niche contains duplicate cell IDs.')
if len(member_names) != int(row['niche_cell_count']) or len(member_names) < TARGET_SIZE:
    raise ValueError('Saved membership disagrees with niche count or target size.')
if CENTER_CELL_NAME not in member_names:
    raise ValueError('The saved niche does not contain the center cell.')

if not h5ad_path.is_file():
    raise FileNotFoundError(f'Slice H5AD not found: {h5ad_path}')
adata = ad.read_h5ad(h5ad_path, backed='r')
try:
    if SPATIAL_KEY not in adata.obsm:
        raise KeyError(f'{SPATIAL_KEY!r} not found. Available keys: {list(adata.obsm)}')
    coords = np.asarray(adata.obsm[SPATIAL_KEY], dtype=float).copy()
    cell_names = pd.Index(adata.obs_names.astype(str))
    obs = adata.obs.copy()
finally:
    adata.file.close()
if not cell_names.is_unique:
    raise ValueError('Slice cell IDs are not unique.')
if coords.ndim != 2 or coords.shape[0] != len(cell_names) or coords.shape[1] < 2:
    raise ValueError('Coordinates must have shape (number of cells, at least 2).')
if not np.isfinite(coords).all():
    raise ValueError('Slice coordinates contain non-finite values.')
member_indices = cell_names.get_indexer(member_names)
if (member_indices < 0).any():
    missing = [name for name, idx in zip(member_names, member_indices) if idx < 0]
    raise ValueError(f'{len(missing)} niche cells missing from slice; examples: {missing[:5]}')
center_index = cell_names.get_loc(CENTER_CELL_NAME)
center_coords = coords[center_index]
coordinate_columns = ['spatial_x', 'spatial_y'] + [f'spatial_dim_{i}' for i in range(2, coords.shape[1])]
print(f'Loaded {len(member_names)} niche cells in a slice of {len(cell_names):,} cells.')
print('Center coordinates:', dict(zip(coordinate_columns, center_coords)))
print('Coordinates use the original H5AD units; the plot uses the first two dimensions.')


In [ ]:
# Export one metrics row, preserving every original CSV column (including JSON arrays).
summary = dict(row)
summary.update(slice_name=SLICE_NAME, spatial_key=SPATIAL_KEY,
               source_metrics_csv=str(metrics_path), source_h5ad=str(h5ad_path))
summary.update({f'center_{name}': value for name, value in zip(coordinate_columns, center_coords)})
summary_df = pd.DataFrame([summary])
center_df = pd.DataFrame([{'center_cell_name': CENTER_CELL_NAME, 'slice_name': SLICE_NAME,
                          **dict(zip(coordinate_columns, center_coords))}])
members_df = pd.DataFrame({'cell_name': member_names, 'slice_name': SLICE_NAME,
                           'is_center': [name == CENTER_CELL_NAME for name in member_names]})
for dim, name in enumerate(coordinate_columns):
    members_df[name] = coords[member_indices, dim]
# Prefix observation columns to avoid overwriting exported identifiers/coordinates.
for name in obs.columns:
    members_df[f'obs_{name}'] = obs.iloc[member_indices][name].to_numpy()
# Older CSVs store the IDs per row; new exports share one file across slices.
if 'parcellation_ids' in row:
    parcellation_ids = json.loads(row['parcellation_ids'])
else:
    with (metrics_path.parent / 'parcellation_ids.json').open(encoding='utf-8') as handle:
        parcellation_ids = json.load(handle)
composition_df = pd.DataFrame({
    'parcellation_id': parcellation_ids,
    'cell_count': [round(fraction * int(row['niche_cell_count']))
                   for fraction in json.loads(row['parcellation_fractions'])],
    'fraction': json.loads(row['parcellation_fractions']),
})
output_dir = resolve_path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
safe_stem = re.sub(r'[^A-Za-z0-9_.-]', '_', f'{SLICE_NAME}_{CENTER_CELL_NAME}_target{TARGET_SIZE}')
output_paths = {}
for suffix, table in [('metrics', summary_df), ('center_coordinates', center_df),
                      ('members', members_df), ('composition', composition_df)]:
    path = output_dir / f'{safe_stem}_{suffix}.csv'
    table.to_csv(path, index=False)
    output_paths[suffix] = path
print('Niche metrics (all original CSV columns are included in the export):')
with pd.option_context('display.max_columns', None, 'display.max_colwidth', 100):
    display(summary_df)
print('Center cell coordinates:')
display(center_df)
print('Parcellation composition:')
display(composition_df)
print('Member cells (preview; the CSV contains all members and observation columns):')
display(members_df.head())


In [ ]:
# Full slice and a close-up of the same saved niche.
xy = coords[:, :2]
niche_xy = xy[member_indices]
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
for ax in axes:
    ax.scatter(xy[:, 0], xy[:, 1], s=2, c='#d0d0d0', alpha=0.5,
               linewidths=0, rasterized=True, label='Other slice cells')
    ax.scatter(niche_xy[:, 0], niche_xy[:, 1], s=14, c='#d62728',
               linewidths=0, label=f'Niche ({len(member_names)} cells)', zorder=3)
    ax.scatter(*center_coords[:2], s=170, marker='*', c='#ffd700',
               edgecolors='black', linewidths=0.9, label='Center cell', zorder=4)
    ax.set(xlabel=f'{SPATIAL_KEY} coordinate 0', ylabel=f'{SPATIAL_KEY} coordinate 1')
    ax.set_aspect('equal', adjustable='box')
low, high = niche_xy.min(axis=0), niche_xy.max(axis=0)
padding = np.maximum((high - low) * 0.15, np.maximum(np.ptp(xy, axis=0) * 0.005, 1e-6))
axes[1].set_xlim(low[0] - padding[0], high[0] + padding[0])
axes[1].set_ylim(low[1] - padding[1], high[1] + padding[1])
axes[0].set_title(f'Full slice: {SLICE_NAME}')
axes[1].set_title(f'Niche close-up | target ≥ {TARGET_SIZE} | reached=True')
axes[1].legend(loc='best')
if INVERT_Y_AXIS:
    for ax in axes:
        ax.invert_yaxis()
fig.suptitle(f'Center cell: {CENTER_CELL_NAME}')
figure_path = output_dir / f'{safe_stem}_niche.png'
fig.savefig(figure_path, dpi=200, bbox_inches='tight')
output_paths['figure'] = figure_path
plt.show()
for name, path in output_paths.items():
    print(f'{name}: {path}')


In [ ]:
# Optional Colab download: bundle the CSVs and figure in one ZIP.
DOWNLOAD_OUTPUTS = False #@param {type:"boolean"}
if DOWNLOAD_OUTPUTS:
    import zipfile
    archive_path = output_dir / f'{safe_stem}_outputs.zip'
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in output_paths.values():
            archive.write(path, arcname=path.name)
    try:
        from google.colab import files
    except ImportError:
        print(f'ZIP saved locally: {archive_path}')
    else:
        files.download(str(archive_path))
